# Stage 2: Instruction Fine-Tuning (AEO/GEO) with Unsloth

This notebook fine-tunes on [data/instruction_dataset.jsonl](../data/instruction_dataset.jsonl), continuing from Stage 1 output when available.

In [1]:
import sys
import subprocess

# Install project dependencies when running in a fresh environment.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 288.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 352.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 431.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0


In [2]:
from pathlib import Path

# Resolve project root for local VS Code execution.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"Using project root: {ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

# Keep cwd at project root and confirm path.
os.chdir(ROOT)
print(os.getcwd())

/content/drive/MyDrive/sft


In [4]:
import json
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "instruction_dataset.jsonl"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
STAGE1_DIR = MODELS_DIR / "stage1_non_instruction_adapter"
STAGE2_DIR = MODELS_DIR / "stage2_instruction_adapter"

BASE_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
OPEN_FALLBACK_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

@dataclass
class SFTConfig:
    max_seq_length: int = 768
    max_train_samples: int = 512
    batch_size: int = 2
    grad_accum: int = 4
    lr: float = 2e-4
    epochs: int = 1
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

cfg = SFTConfig()
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(cfg)
print(f"Instruction dataset path: {DATA_PATH}")
print(f"Stage1 path exists: {STAGE1_DIR.exists()} -> {STAGE1_DIR}")

W0712 12:01:04.056000 7126 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0712 12:01:04.108000 7126 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


SFTConfig(max_seq_length=768, max_train_samples=512, batch_size=2, grad_accum=4, lr=0.0002, epochs=1, lora_r=16, lora_alpha=16, lora_dropout=0.0)
Instruction dataset path: /content/drive/MyDrive/sft/data/instruction_dataset.jsonl
Stage1 path exists: True -> /content/drive/MyDrive/sft/models/stage1_non_instruction_adapter


In [5]:
def load_instruction_rows(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            item = json.loads(line)
            if not isinstance(item.get("instruction"), str):
                raise ValueError(f"Line {i}: missing instruction string")
            if not isinstance(item.get("response"), str):
                raise ValueError(f"Line {i}: missing response string")
            rows.append(item)
    return rows


def format_sft_text(example):
    return {
        "text": (
            "You are an internal enterprise AEO/GEO assistant. "
            "Provide accurate, practical, structured answers.\n\n"
            f"Instruction: {example['instruction']}\n"
            f"Response: {example['response']}"
        )
    }


rows = load_instruction_rows(DATA_PATH)
formatted = [format_sft_text(r) for r in rows]
train_ds = Dataset.from_list(formatted)
if cfg.max_train_samples and len(train_ds) > cfg.max_train_samples:
    train_ds = train_ds.shuffle(seed=SEED).select(range(cfg.max_train_samples))

print(f"Loaded rows: {len(rows)}")
print(f"Train rows used: {len(train_ds)}")
print(train_ds[0]["text"][:400])

Loaded rows: 250
Train rows used: 250
You are an internal enterprise AEO/GEO assistant. Provide accurate, practical, structured answers.

Instruction: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Response: Saint Bernadette Soubirous


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

SAFE_FALLBACK_MODEL_IDS = [
    "Qwen/Qwen2-0.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "distilgpt2",
]


def _can_use_bitsandbytes_4bit() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        import bitsandbytes as bnb
        from bitsandbytes.cextension import lib
        if not getattr(lib, "compiled_with_cuda", False):
            print("bitsandbytes CUDA backend unavailable; disabling 4-bit path.")
            return False
        if getattr(lib, "cdequantize_blockwise_fp32", None) is None:
            print("bitsandbytes missing required kernels; disabling 4-bit path.")
            return False
        _ = bnb.__version__  # touch import for clearer failure in broken installs
        return True
    except Exception as exc:
        print(f"bitsandbytes unavailable ({exc}); disabling 4-bit path.")
        return False


use_4bit = _can_use_bitsandbytes_4bit()


def _is_stage1_adapter_only(path: Path) -> bool:
    return (path / "adapter_config.json").exists() and not (path / "config.json").exists()


def _pick_lora_target_modules(current_model):
    module_names = [name for name, _ in current_model.named_modules()]
    if any(name.endswith("q_proj") for name in module_names):
        return ["q_proj", "k_proj", "v_proj", "o_proj"]
    if any(name.endswith("c_attn") for name in module_names):
        return ["c_attn", "c_proj"]
    # Last-resort generic attention projections for uncommon architectures.
    fallback = [
        name.split(".")[-1]
        for name in module_names
        if name.split(".")[-1] in {"query", "key", "value", "dense"}
    ]
    unique_fallback = sorted(set(fallback))
    if unique_fallback:
        return unique_fallback
    raise RuntimeError("Could not infer LoRA target modules for selected model.")


def load_base_model_and_tokenizer():
    candidate_models = []
    if STAGE1_DIR.exists():
        if _is_stage1_adapter_only(STAGE1_DIR):
            print("Stage1 directory is adapter-only; skipping direct base-model load from Stage1 path.")
        else:
            candidate_models.append(str(STAGE1_DIR))

    ordered_candidates = [BASE_MODEL_ID, OPEN_FALLBACK_MODEL_ID, *SAFE_FALLBACK_MODEL_IDS]
    if not use_4bit:
        ordered_candidates = [m for m in ordered_candidates if "bnb-4bit" not in m.lower()]

    for model_name in ordered_candidates:
        if model_name not in candidate_models:
            candidate_models.append(model_name)

    tokenizer = None
    model = None
    selected = None

    for model_name in candidate_models:
        try:
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            except Exception:
                tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model_kwargs = {
                "torch_dtype": compute_dtype,
                "device_map": "auto" if torch.cuda.is_available() else None,
            }

            model = None
            if use_4bit and "bnb-4bit" in model_name.lower():
                try:
                    quant_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_use_double_quant=True,
                        bnb_4bit_compute_dtype=torch.float16,
                    )
                    model = AutoModelForCausalLM.from_pretrained(
                        model_name,
                        quantization_config=quant_config,
                        **model_kwargs,
                    )
                except Exception as q_exc:
                    print(f"4-bit load failed for {model_name}: {q_exc}")

            if model is None:
                model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

            selected = model_name
            break
        except Exception as exc:
            print(f"Model load failed for {model_name}: {exc}")

    if model is None or tokenizer is None:
        attempted = ", ".join(candidate_models)
        raise RuntimeError(f"Could not load any candidate model for Stage 2. Tried: {attempted}")

    lora_targets = _pick_lora_target_modules(model)
    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=lora_targets,
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )
    model = get_peft_model(model, lora_cfg)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Selected model: {selected}")
    print(f"LoRA targets: {lora_targets}")
    print(f"Trainable params: {trainable:,} / {total:,}")

    return model, tokenizer, selected


model, tokenizer, model_name_used = load_base_model_and_tokenizer()

def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=cfg.max_seq_length,
        padding=False,
    )

tokenized_train_ds = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names,
    desc="Tokenizing instruction dataset",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "stage2_runs"),
    per_device_train_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.lr,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

bitsandbytes CUDA backend unavailable; disabling 4-bit path.
Stage1 directory is adapter-only; skipping direct base-model load from Stage1 path.


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Selected model: Qwen/Qwen2.5-0.5B-Instruct
LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
Trainable params: 2,162,688 / 496,195,456


Tokenizing instruction dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

/tmp/ipykernel_7126/832352927.py:173: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [9]:
trainer.train()

STAGE2_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(STAGE2_DIR))
tokenizer.save_pretrained(str(STAGE2_DIR))
print(f"Saved Stage 2 adapter/model to {STAGE2_DIR}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.633700
20,2.268200
30,1.604400


Saved Stage 2 adapter/model to /content/drive/MyDrive/sft/models/stage2_instruction_adapter


In [12]:

questions = [
    "We are launching a new enterprise SaaS tool. How should we structure landing page text and data tables to maximize citation chances in Gemini and ChatGPT Search?",
    "Review an FAQ page with Speakable and Dataset schema. Which microdata fields are commonly missed for AI Overviews and voice assistants?",
    "Perplexity mentions dropped 15 percent this quarter. What framework helps diagnose crawler blocks vs stale stats vs competitor GEO improvements?",
    "Rewrite a blog paragraph using GEO principles with authoritative citations and high information density.",
    "How do I design answer blocks for passage-level retrieval in answer engines?",
    "What authority signals make a brand more citeable in generative search outputs?",
    "How should we measure Share of Model for our brand across major answer engines?",
    "How should we structure comparison tables so LLM retrievers can extract key attributes correctly?",
    "How can we improve entity salience for product pages in AEO and GEO?",
    "How do we balance concise direct answers with deeper context for complex B2B questions?",
]


def load_base_eval_model():
    # Evaluate base independently from SFT adapter and avoid bnb-only checkpoints when 4-bit is unavailable.
    eval_candidates = [OPEN_FALLBACK_MODEL_ID, BASE_MODEL_ID, *SAFE_FALLBACK_MODEL_IDS]
    if not use_4bit:
        eval_candidates = [m for m in eval_candidates if "bnb-4bit" not in m.lower()]

    tried = []
    for model_name in eval_candidates:
        if model_name in tried:
            continue
        tried.append(model_name)
        try:
            try:
                tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            except Exception:
                tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            mdl = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
            )
            return mdl, tok, model_name
        except Exception as exc:
            print(f"Base eval load failed for {model_name}: {exc}")

    raise RuntimeError(f"Unable to load a base model for comparison. Tried: {', '.join(tried)}")


def generate_answer(current_model, current_tokenizer, question, max_new_tokens=180):
    prompt = (
        "You are an AEO and GEO assistant. Answer clearly and practically.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = current_tokenizer(prompt, return_tensors="pt")
    try:
        model_device = current_model.device
    except Exception:
        model_device = next(current_model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}
    with torch.no_grad():
        out = current_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=current_tokenizer.eos_token_id,
            pad_token_id=current_tokenizer.pad_token_id,
        )
    text = current_tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("Answer:", 1)[-1].strip() if "Answer:" in text else text.strip()


def pick_better(base_ans: str, sft_ans: str):
    sft_score = 0
    base_score = 0

    sft_lower = sft_ans.lower()
    base_lower = base_ans.lower()

    domain_terms = ["schema", "citation", "entity", "retrieval", "table", "authority", "share of model"]
    if any(term in sft_lower for term in domain_terms):
        sft_score += 2
    if any(term in base_lower for term in domain_terms):
        base_score += 2

    if len(sft_ans.split()) >= 35:
        sft_score += 1
    if len(base_ans.split()) >= 35:
        base_score += 1

    if any(x in sft_lower for x in ["step", "framework", "checklist", "measure"]):
        sft_score += 1
    if any(x in base_lower for x in ["step", "framework", "checklist", "measure"]):
        base_score += 1

    if sft_score >= base_score:
        return "Fine-Tuned Model", "More domain-specific, clearer, and less generic under evaluation criteria."
    return "Base Model", "Base response appears more correct/helpful for this specific prompt."


base_model_eval, base_tok_eval, base_name = load_base_eval_model()
comparisons = []

for q in questions:
    base_ans = generate_answer(base_model_eval, base_tok_eval, q)
    sft_ans = generate_answer(model, tokenizer, q)
    better, reason = pick_better(base_ans, sft_ans)
    comparisons.append(
        {
            "Question": q,
            "Base Model Answer": base_ans,
            "Fine-Tuned Model Answer": sft_ans,
            "Which is Better?": better,
            "Reason": reason,
        }
    )

comparison_df = pd.DataFrame(comparisons)
comparison_df

report_path = REPORTS_DIR / "sft_model_comparison.md"
with report_path.open("w", encoding="utf-8") as f:
    f.write("# SFT Model Comparison\n\n")
    f.write(f"Base model used: {base_name}\\n")
    f.write(f"SFT starting model used: {model_name_used}\\n\\n")
    f.write("Evaluation criteria: correctness, domain accuracy, clarity, safety, helpfulness, less generic response, domain-specific behavior.\\n\\n")
    f.write("| Question | Base Model Answer | Fine-Tuned Model Answer | Which is Better? | Reason |\\n")
    f.write("|---|---|---|---|---|\\n")
    for row in comparisons:
        q = row["Question"].replace("|", "\\|").replace("\n", " ")
        b = row["Base Model Answer"].replace("|", "\\|").replace("\n", " ")
        s = row["Fine-Tuned Model Answer"].replace("|", "\\|").replace("\n", " ")
        w = row["Which is Better?"].replace("|", "\\|").replace("\n", " ")
        r = row["Reason"].replace("|", "\\|").replace("\n", " ")
        f.write(f"| {q} | {b} | {s} | {w} | {r} |\\n")

print(f"Wrote {report_path}")

Wrote /content/drive/MyDrive/sft/reports/sft_model_comparison.md
